# 📝 Homework Day 1: Data Engineering Pipeline
## Agentic RAG: From Zero to Hero

---

### 📋 Instructions

1. **Complete this by yourself** — do not use AI to help write code
2. **No copying** — each person's data will be **different** (generated from the student ID)
3. **Submit this notebook** together with the executed results (.ipynb)
4. **Score**: 10 points

> ⚠️ **The system will detect plagiarism** using hash values, embeddings, and scores that must match each student's ID

> **🇹🇭 Thai Text in This Notebook**
>
> Sample data and queries are in Thai because this workshop teaches Thai RAG. English translations are provided as inline comments (`# "translation"`).

## 📦 Install Dependencies

In [ ]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'rank_bm25', 'pythainlp', 'scikit-learn']:
    _pip_install(_pkg)

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ Installation complete!')

## 🎓 Fill in Student Information

In [ ]:
# ─── Fill in your information ───
STUDENT_NAME = ''   # เช่น 'สมชาย ใจดี'  # "   # e.g. "
STUDENT_ID   = ''   # เช่น '6512345678'  # "   # e.g. "
PHONE        = ''   # เช่น '081-234-5678'  # "   # e.g. "
LINE_ID      = ''   # เช่น 'somchai.j'  # "   # e.g. "

# ─── Validation (do not edit) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'  # "❌ Please enter your student ID!"
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'  # "❌ Please enter your full name!"

print(f'✅ Full name: {STUDENT_NAME}')
print(f'✅ Student ID: {STUDENT_ID}')
print(f'📱 Phone number: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 Step 2: Create Your Personalized Dataset (Do not edit this cell)

In [ ]:
%%time
# ===== Do not edit this cell =====
# Create a personalized dataset from the student ID

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

# Main content — shuffle order and select content based on the student ID
all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ โดยไม่จำเป็นต้องเขียนโปรแกรมอย่างชัดเจนสำหรับทุกกรณี',  # "Machine Learning is a subfield of artificial in..."
    'Deep Learning เป็นเทคนิคหนึ่งของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา และการสร้างข้อความ',  # "Deep Learning is a technique within Machine Lea..."
    'Natural Language Processing หรือ NLP คือสาขาที่เกี่ยวข้องกับการทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์ การสรุปข้อความ และการตอบคำถาม',  # "Natural Language Processing or NLP is the field..."
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้อง ทันสมัย และอ้างอิงแหล่งข้อมูลได้',  # "Retrieval Augmented Generation or RAG is a tech..."
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้สามารถค้นหาข้อมูลที่มีความหมายคล้ายคลึงกันได้อย่างรวดเร็ว',  # "Vector Database is a database designed to store..."
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลขหรือ Vector ที่สามารถแสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้คอมพิวเตอร์สามารถเปรียบเทียบความคล้ายคลึงระหว่างข้อความได้',  # "Text Embedding is the process of converting tex..."
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูลแบบลำดับ เป็นพื้นฐานของโมเดลภาษาขนาดใหญ่อย่าง GPT BERT และ Gemini',  # "Transformer is a neural network architecture th..."
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่งหรือ Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพและความแม่นยำของคำตอบอย่างมาก',  # "Prompt Engineering is the discipline of designi..."
    'Fine-tuning คือกระบวนการปรับแต่งโมเดลที่ผ่านการฝึกมาแล้วด้วยข้อมูลเฉพาะทางเพิ่มเติม เพื่อให้โมเดลทำงานได้ดีขึ้นในงานที่เฉพาะเจาะจง เช่น การวิเคราะห์เอกสารทางการแพทย์',  # "Fine-tuning is the process of adapting a pre-tr..."
    'Tokenization คือกระบวนการแบ่งข้อความออกเป็นหน่วยย่อยที่เรียกว่า Token ซึ่งอาจเป็นคำ คำย่อย หรือตัวอักษร สำหรับภาษาไทยการตัดคำมีความซับซ้อนเพราะไม่มีเว้นวรรคระหว่างคำ',  # "Tokenization is the process of splitting text i..."
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic Chunking',  # "Chunking is the process of splitting a long doc..."
    'Cosine Similarity เป็นวิธีวัดความคล้ายคลึงระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน ค่า 0 หมายถึงตั้งฉาก นิยมใช้ในงาน NLP และ Information Retrieval',  # "Cosine Similarity is a method for measuring sim..."
]

# Shuffle order based on the student's seed
random.shuffle(all_paragraphs)

# Select 8 paragraphs + create 1 duplicate
selected = all_paragraphs[:8]
duplicate_idx = random.randint(0, 5)
selected.append(selected[duplicate_idx])  # ย่อหน้าที่ซ้ำ

# Save as files
os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ Personalized data for {STUDENT_ID} has been created successfully!')
print(f'📁 Number of files: {len(selected)} files (1 duplicate file included)')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} characters)')

---
## 🎯 Task: Build a Data Engineering Pipeline

Build a **RAG Data Pipeline** from your personalized dataset by following the steps below.
Each step has **answers that must be reported** — these values will be different for each student.

---

### Step 1: Detect Duplicates with MD5 (2 points)

- Compute the MD5 hash of every file in `homework_data/`
- Find the duplicate files
- Remove the duplicate file(s) (keep the first one)

**📝 Report:**
1. Which pair of files is duplicated?
2. What is the MD5 hash of the duplicate file?
3. How many files remain after removal?

In [ ]:
# Step 1: Write your duplicate detection code here

# 💡 Hint:
#   1. import hashlib
#   2. Create a function compute_md5(filepath) that reads a file and returns the hash
#   3. Loop through every file in 'homework_data/' and store hashes in a dict
#   4. If a hash repeats → that file is a duplicate



# ✅ Self-check (run after finishing your code)
# assert dup_hash is not None, '❌ You have not found the duplicate file yet'
# assert files_remaining >= 7, '❌ The number of files after deletion should be greater than 7'
# print(f'✅ Step 1 passed: duplicate files={dup_files}, MD5={dup_hash}')

### Step 2: Chunking (2 points)

- Combine the text from the remaining files (excluding duplicates)
- Chunk using `RecursiveCharacterTextSplitter` with `chunk_size=150`, `chunk_overlap=30`

**📝 Report:**
1. How many chunks are there in total?
2. What is the content of chunk 1? (copy and paste it)
3. How many characters are in the shortest chunk?

In [ ]:
# Step 2: Write your chunking code here

# 💡 Hint:
#   1. from langchain_text_splitters import RecursiveCharacterTextSplitter
#   2. Combine content from non-duplicate files
#   3. Create a splitter with chunk_size=150, chunk_overlap=30
#   4. Use splitter.split_text(all_text)



# ✅ Self-check
# assert len(chunks) > 0, '❌ You have not chunked the text yet'
# assert len(chunks[0]) <= 150, '❌ The chunk is too large'
# print(f'✅ Step 2 passed: {len(chunks)} chunks, chunk 1 = {len(chunks[0])} chars')

### Step 3: Embedding + Similarity (3 points)

- Create embeddings using `intfloat/multilingual-e5-large`
- Compute **Cosine Similarity** between the query and every chunk
- Query: `'query: techniques for retrieving semantically similar information'`

**📝 Report:**
1. Which chunk has the highest similarity? (specify the chunk number and score to 4 decimal places)
2. Which chunk has the lowest similarity? (specify the chunk number and score to 4 decimal places)
3. Based on the result, why is that chunk the most similar? (Explain in your own words in 2–3 sentences)

In [ ]:
# Step 3: Write your embedding + similarity code here
# Don't forget to add the 'query: ' and 'passage: ' prefixes as taught in class

# 💡 Hint:
#   1. from sentence_transformers import SentenceTransformer
#   2. model = SentenceTransformer('intfloat/multilingual-e5-large')
#   3. query = 'query: techniques for retrieving semantically similar information'
#   4. passages = ['passage: ' + c for c in chunks]
#   5. query_emb = model.encode(query)
#   6. passage_embs = model.encode(passages)
#   7. Use cosine_similarity() from sklearn



# ✅ Self-check
# assert best_score > 0.5, '❌ similarity score is too low, check the prefixes'
# print(f'✅ Step 3 passed: best chunk={best_idx}, score={best_score:.4f}')

### Step 4: Qdrant Retrieval (3 points)

- Create a Qdrant client (in-memory) + a collection named `f'hw_{STUDENT_ID}'`
- Upsert every chunk + embedding into Qdrant
- Search with the query: `'query: splitting text into smaller parts'`
- Use `top_k=3`

**📝 Report:**
1. For the Top-3 results, what is the score of each rank? (4 decimal places)
2. What is the top-ranked result about?
3. Write a summary: if this system were to be used in a real-world setting, what should be improved? (Explain in 3–5 sentences)

In [ ]:
# Step 4: Write your Qdrant retrieval code here

# 💡 Hint:
#   1. from qdrant_client import QdrantClient, models
#   2. qdrant = QdrantClient(':memory:')
#   3. Create a collection named f'hw_{STUDENT_ID}'
#   4. Upsert every chunk + embedding into Qdrant
#   5. query = 'query: splitting text into smaller parts'
#   6. qdrant.query_points(..., limit=3)



# ✅ Self-check
# assert len(results) == 3, '❌ You should get top_k=3 results'
# print(f'✅ Step 4 passed: Top-3 scores={[r.score for r in results]}')

## 📊 Grading Criteria

| Step | Score | Criteria |
|---------|:-----:|------|
| 1. Find Duplicates | 2 | `find_duplicates` function works correctly, results are correct |
| 2. Chunking | 3 | chunk_size/overlap are correct, methods are compared |
| 3. Embedding + Search | 3 | embedding works, search works, results are reasonable |
| 4. Result Analysis | 2 | explains output quality, compares parameters |
| **Total** | **10** | |

---
## ✅ Verify Your Answers

Run the cell below to generate a **Verification Code** for submission

In [ ]:
# ===== Do not edit this cell =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_day1_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 Full name: {STUDENT_NAME}')
print(f'🎓 Student ID: {STUDENT_ID}')
print(f'📱 Phone number: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 Submit before: _________________ (set by instructor)')
print('=' * 50)
print()
print('📋 Checklist before submission:')
print('  [ ] Filled in all personal information completely')
print('  [ ] Step 1: Identified duplicate file(s) + MD5 hash')
print('  [ ] Step 2: Specified the number of chunks + content of chunk 1')
print('  [ ] Step 3: Specified the most similar/least similar chunk + explanation')
print('  [ ] Step 4: Specified Top-3 scores + improvement summary')
print('  [ ] All cells have been run and show outputs')